In [2]:
# Spline cúbica interpolante

def gauss_solve(A, b):
    """
    Resolve o sistema linear A x = b por eliminação de Gauss simples.
    """
    n = len(b)

    # Eliminação para transformar A em triangular superior
    for i in range(n):
        pivot = A[i][i]
        if abs(pivot) < 1e-14:
            raise ValueError("Pivô zero – exemplo patológico para esse solver simples.")

        # Normaliza linha
        for j in range(i, n):
            A[i][j] /= pivot
        b[i] /= pivot

        # Zera abaixo da diagonal
        for k in range(i + 1, n):
            fator = A[k][i]
            for j in range(i, n):
                A[k][j] -= fator * A[i][j]
            b[k] -= fator * b[i]

    # Retrossubstituição
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        s = sum(A[i][j] * x[j] for j in range(i + 1, n))
        x[i] = b[i] - s

    return x


def natural_cubic_spline(xs, ys, var="x"):
    """
    Constrói a spline cúbica interpolante natural para pontos (xs, ys).
    Retorna a lista de coeficientes [ (a_k, b_k, c_k, d_k, x_k, x_{k+1}) ].

    Cada intervalo [x_k, x_{k+1}] é descrito por:
        s_k(x) = a_k (x - x_k)^3 + b_k (x - x_k)^2 + c_k (x - x_k) + d_k
    """

    n = len(xs) - 1           # número de subintervalos
    h = [xs[i+1] - xs[i] for i in range(n)]

    print("=== Spline cúbica natural ===\n")
    print("Nós x_k:", xs)
    print("Valores y_k:", ys)
    print("Passos h_k:", h, "\n")

    # ===== 1) Montar o sistema A g = b para g1..g_{n-1} =====
    m = n - 1                 # quantidade de g internos (g_1..g_{n-1})
    if m <= 0:
        raise ValueError("Precisa de pelo menos 3 pontos para spline cúbica.")

    A = [[0.0] * m for _ in range(m)]
    b = [0.0] * m

    print("Montando o sistema A g = b (condição natural: g0 = gn = 0)\n")

    for k in range(1, n):     # k = 1..n-1 (índices dos nós internos)
        i = k - 1             # índice da linha em A/b (0..m-1)
        h_km1 = h[k-1]
        h_k = h[k]

        # Diagonal principal
        A[i][i] = 2 * (h_km1 + h_k)

        # Subdiagonal
        if i > 0:
            A[i][i-1] = h_km1

        # Superdiagonal
        if i < m - 1:
            A[i][i+1] = h_k

        # Termo independente
        b[i] = 6 * (
            (ys[k+1] - ys[k]) / h_k
            - (ys[k] - ys[k-1]) / h_km1
        )

        print(f"Equação para g_{k}:")
        linha = []
        if i > 0:
            linha.append(f"{h_km1} * g_{k-1}")
        linha.append(f"2({h_km1}+{h_k}) * g_{k}")
        if i < m - 1:
            linha.append(f"{h_k} * g_{k+1}")
        print("  " + " + ".join(linha) + f" = {b[i]}")
        print()

    print("Matriz A:")
    for row in A:
        print("  ", row)
    print("Vetor b:", b, "\n")

    # ===== 2) Resolver o sistema para g1..g_{n-1} =====
    g_internos = gauss_solve([row[:] for row in A], b[:])
    g = [0.0] + g_internos + [0.0]  # natural: g0 = gn = 0

    print("Soluções g_k (segundas derivadas nos nós):")
    for i, gi in enumerate(g):
        print(f"g_{i} = {gi}")
    print()

    # ===== 3) Calcular coeficientes (a_k, b_k, c_k, d_k) em cada intervalo =====
    coeficientes = []

    print("Coeficientes dos polinômios em cada intervalo [x_k, x_{k+1}]:\n")

    for k in range(n):
        hk = h[k]
        a_k = (g[k+1] - g[k]) / (6 * hk)
        b_k = g[k] / 2.0
        c_k = (ys[k+1] - ys[k]) / hk - (2*hk*g[k] + hk*g[k+1]) / 6.0
        d_k = ys[k]

        coeficientes.append((a_k, b_k, c_k, d_k, xs[k], xs[k+1]))

        print(f"Intervalo {k+1}: [{xs[k]}, {xs[k+1]}]")
        print("s_{%d}(x) = a_k (x - x_k)^3 + b_k (x - x_k)^2 + c_k (x - x_k) + d_k" % (k+1))
        print(f"  a_k = (g_{k+1} - g_{k})/(6 h_k) = ({g[k+1]} - {g[k]})/(6*{hk}) = {a_k}")
        print(f"  b_k = g_{k}/2 = {g[k]}/2 = {b_k}")
        print(f"  c_k = (y_{k+1}-y_{k})/h_k - (2 h_k g_{k} + h_k g_{k+1})/6")
        print(f"      = ({ys[k+1]} - {ys[k]})/{hk} - (2*{hk}*{g[k]} + {hk}*{g[k+1]})/6 = {c_k}")
        print(f"  d_k = y_{k} = {d_k}")
        print(f"\n=> s_{k+1}(x) = {a_k}*({var} - {xs[k]})**3"
              f" + {b_k}*({var} - {xs[k]})**2"
              f" + {c_k}*({var} - {xs[k]})"
              f" + {d_k}\n")
        print("---------------------------------------\n")

    return coeficientes


def avaliar_spline_cubica(coeficientes, x):
    """
    Avalia a spline cúbica num ponto x dado.
    coeficientes é a lista retornada por natural_cubic_spline.
    """
    for a, b, c, d, xk, xk1 in coeficientes:
        if xk <= x <= xk1:
            t = x - xk
            return ((a*t + b)*t + c)*t + d
    # se x estiver fora do intervalo, pode extrapolar usando o primeiro/último
    if x < coeficientes[0][4]:
        a, b, c, d, xk, xk1 = coeficientes[0]
        t = x - xk
        return ((a*t + b)*t + c)*t + d
    else:
        a, b, c, d, xk, xk1 = coeficientes[-1]
        t = x - xk
        return ((a*t + b)*t + c)*t + d


def verificar_spline_cubica(xs, ys, coeficientes):
    """
    Verifica se a spline passa exatamente pelos pontos (xs, ys).
    """
    print("=== Verificando spline cúbica nos nós ===")
    ok = True
    for xi, yi in zip(xs, ys):
        val = avaliar_spline_cubica(coeficientes, xi)
        status = "OK ✅" if abs(val - yi) < 1e-6 else "ERRO ❌"
        print(f"x = {xi}: S(x) = {val}, esperado = {yi} → {status}")
        if abs(val - yi) >= 1e-6:
            ok = False
    if ok:
        print("\nSpline cúbica está CORRETA em todos os nós")
    else:
        print("\nSpline cúbica NÃO interpola corretamente algum nó.")
    return ok


In [4]:
xs = [0.0, 0.5, 1.0, 1.5, 2.0]
ys = [3.0, 1.8616, -0.5571, -4.1987, -9.0536]

coef = natural_cubic_spline(xs, ys)
verificar_spline_cubica(xs, ys, coef)

# Aproximação de f(0.25) que o slide calcula
x0 = 0.25
print("\nAproximação f(0.25) =", avaliar_spline_cubica(coef, x0))


=== Spline cúbica natural ===

Nós x_k: [0.0, 0.5, 1.0, 1.5, 2.0]
Valores y_k: [3.0, 1.8616, -0.5571, -4.1987, -9.0536]
Passos h_k: [0.5, 0.5, 0.5, 0.5] 

Montando o sistema A g = b (condição natural: g0 = gn = 0)

Equação para g_1:
  2(0.5+0.5) * g_1 + 0.5 * g_2 = -15.363599999999998

Equação para g_2:
  0.5 * g_1 + 2(0.5+0.5) * g_2 + 0.5 * g_3 = -14.674799999999996

Equação para g_3:
  0.5 * g_2 + 2(0.5+0.5) * g_3 = -14.559600000000003

Matriz A:
   [2.0, 0.5, 0.0]
   [0.5, 2.0, 0.5]
   [0.0, 0.5, 2.0]
Vetor b: [-15.363599999999998, -14.674799999999996, -14.559600000000003] 

Soluções g_k (segundas derivadas nos nós):
g_0 = 0.0
g_1 = -6.6540857142857135
g_2 = -4.1108571428571405
g_3 = -6.252085714285716
g_4 = 0.0

Coeficientes dos polinômios em cada intervalo [x_k, x_{k+1}]:

Intervalo 1: [0.0, 0.5]
s_{1}(x) = a_k (x - x_k)^3 + b_k (x - x_k)^2 + c_k (x - x_k) + d_k
  a_k = (g_1 - g_0)/(6 h_k) = (-6.6540857142857135 - 0.0)/(6*0.5) = -2.218028571428571
  b_k = g_0/2 = 0.0/2 = 0.0
  c_k